In [0]:
scope = "keyvaultcapstone"

kafka_key = dbutils.secrets.get(scope, "kafkaapikey")
kafka_secret = dbutils.secrets.get(scope, "kafkaapisecret")
kafka_server = dbutils.secrets.get(scope, "kafkabootstrapserver")

print("Secrets loaded successfully")

In [0]:
jaas = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="{kafka_key}" password="{kafka_secret}";'
)

print("Kafka authentication configured")

In [0]:
topic = "inventory_topic"

df_kafka_raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_server)
    .option("subscribe", topic) # Replace with your actual topic name
    .option("startingOffsets", "latest") # Use 'latest' if you only want new events
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas)
    .load()
)

print("Kafka streaming reader created")

In [0]:
df_bronze_inventory = df_kafka_raw.selectExpr(
    "CAST(value AS STRING) AS message_payload",
    "topic",
    "partition",
    "offset",
    "timestamp AS event_timestamp"
)

In [0]:
catalog_name = 'mavenmarket'
schema_name = 'bronze'
table_name = 'kafka_inventory'
checkpoint_path = f"/Volumes/{catalog_name}/{schema_name}/checkpoints/{table_name}"

In [0]:
query = (df_bronze_inventory.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="30 seconds") 
    .table(f"{catalog_name}.{schema_name}.{table_name}")
)